<a id="resumo"></a>
# Resumo

Modelos de Machine Learning implantados em produção frequentemente perdem desempenho ao longo do tempo devido ao **concept drift**: a mudança na relação estatística entre variáveis de entrada e a variável-alvo. Em séries temporais meteorológicas, esse fenômeno é particularmente relevante, pois sazonalidade, mudanças de regime climático e ruído observacional alteram continuamente a distribuição dos dados. Este estudo investiga como diferentes políticas de atualização de modelo lidam com essa não estacionariedade em um problema real: a previsão binária de chuva horária no Aeroporto Internacional de Miami, utilizando 14 anos de observações da rede ASOS (2012–2025).

Foram comparadas três estratégias sob condições experimentais idênticas. Mesmas janelas temporais, mesmos dados e mesmo processo de avaliação: (i) **aprendizado incremental**, com atualização contínua do modelo via aprendizado online; (ii) **retreinamento periódico**, com recriação do modelo a cada nova janela; e (iii) **treinamento único**, uma baseline estática. A avaliação seguiu um esquema de janelas deslizantes (180 dias de treino, 14 de teste, passo de 14 dias), preservando a sequência temporal. Dado o desbalanceamento da classe positiva, a métrica principal adotada foi o **F1-score**, complementada por precisão, recall e acurácia.

Os resultados mostram que o cenário **Incremental** obteve o maior F1 agregado (0,470), superando o Retreinamento (0,381) e o Treinamento único (0,333), com maior regularidade ao longo dos 14 anos avaliados. O teste de Wilcoxon pareado, com correção de Bonferroni, confirmou diferenças estatisticamente significativas e de efeito grande entre o cenário Incremental e os demais. Conclui-se que, em problemas temporais reais como a previsão de chuva, a política de atualização do modelo é parte central da solução, e não um detalhe de engenharia secundário.

References

- J. Gama, I. Žliobaitė, A. Bifet, M. Pechenizkiy, and A. Bouchachia. A Survey on Concept Drift Adaptation. ACM Computing Surveys, 46(4), Article 44, 2014. DOI: 10.1145/2523813.

- J. Lu, A. Liu, F. Dong, F. Gu, J. Gama, and G. Zhang. Learning under Concept Drift: A Review. IEEE Transactions on Knowledge and Data Engineering, 31(12), 2346–2363, 2019. DOI: 10.1109/TKDE.2018.2876857.

- Iowa Environmental Mesonet (IEM), Iowa State University. ASOS / METAR data download interface. Available at: https://mesonet.agron.iastate.edu/request/download.phtml.


# Sumário

- [1. Introdução](#sec-introducao)
  - [Objetivos](#sec-objetivos)
  - [1.2 Problema e Hipótese](#sec-problema-hipotese)
  - [1.2.1 Por que F1-score como Métrica Principal](#sec-f1)
- [2. Metodologia](#sec-metodologia)
  - [2.1 Cenários Comparados](#sec-cenarios)
  - [2.2 Ambiente e Escolhas Metodológicas](#sec-ambiente)
  - [2.3 Dataset e Pré-processamento](#sec-dataset)
    - [2.3.1 Preparação dos Dados](#sec-preparacao-dados)
    - [2.3.2 Estratégia de Validação Temporal](#sec-validacao-temporal)
  - [2.4 Métricas de Avaliação](#sec-metricas)
  - [2.5 Estrutura dos Resultados](#sec-estrutura-resultados)
  - [2.6 Cenário 1 — Aprendizado Incremental](#sec-incremental)
  - [2.7 Cenário 2 — Retreinamento Periódico](#sec-retreinamento)
  - [2.8 Cenário 3 — Treinamento Único](#sec-treinamento-unico)
- [3. Execução do Experimento](#sec-execucao)
- [4. Resultados](#sec-resultados)
  - [4.1 Dinâmica Temporal da Variável-Alvo](#sec-dinamica-alvo)
  - [4.2 Consolidação dos Resultados](#sec-consolidacao)
  - [4.3 Leitura dos Resultados Agregados](#sec-resultados-agregados)
  - [4.4 Comparação Estatística entre Cenários](#sec-comparacao-estatistica)
  - [4.5 Interpretação Estatística](#sec-interpretacao-estatistica)
- [5. Discussão](#sec-discussao)
  - [5.1 Limitações do Experimento](#sec-limitacoes)
  - [5.2 Principais Resultados](#sec-principais-resultados)
- [6. Conclusão](#sec-conclusao)

<a id="sec-introducao"></a>
# 1. Introdução

Modelos de Machine Learning em produção enfrentam um problema recorrente: o mundo muda, mas o modelo treinado continua o mesmo. Esse fenômeno, conhecido como **concept drift**, ocorre quando a relação estatística entre as variáveis de entrada e o alvo se altera ao longo do tempo. Seja por sazonalidade, mudanças de regime climático, ruído de sensores ou qualquer outro fator não estacionário.

Em séries temporais meteorológicas isso é especialmente relevante: um modelo treinado com os padrões de um período pode perder desempenho meses depois, simplesmente porque a distribuição dos dados evoluiu. A pergunta prática que motiva este estudo é: **qual política de atualização de modelo lida melhor com essa mudança ao longo do tempo?**

Este notebook investiga essa questão em um problema real, previsão binária de chuva horária no Aeroporto Internacional de Miami. Comparando três estratégias de atualização de modelo sob exatamente as mesmas condições experimentais.

<a id="sec-objetivos"></a>
## Objetivos

- Comparar três políticas de atualização de modelo (estática, retreinamento periódico e aprendizado incremental) em um problema real de série temporal.
- Avaliar não apenas o desempenho agregado, mas a **estabilidade ao longo do tempo** de cada estratégia.
- Confirmar as diferenças observadas com um teste estatístico pareado.


In [1]:
# Dependência opcional: descomente a linha abaixo caso o pacote "river"
# ainda não esteja instalado no ambiente de execução.
# !{sys.executable} -m pip install river -q

In [2]:
# Bibliotecas padrão
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple
from IPython.display import display

In [3]:
# Bibliotecas de terceiros
import numpy as np
import pandas as pd
from river import compose, linear_model, optim, preprocessing
from scipy.stats import wilcoxon
from sklearn.metrics import confusion_matrix
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.graph_objects as go



In [4]:
#Ajustes gráficos

# Paleta de cores consistente por cenário, reutilizada em todas as figuras.
SCENARIO_COLORS: Dict[str, str] = {
    "Incremental": "#2ca02c",        # verde
    "Retreinamento": "#1f77b4",      # azul
    "Treinamento único": "#d62728",  # vermelho
}


def apply_layout(
    fig: go.Figure,
    title: str,
    height: int = 450,
    width: int = 950,
) -> go.Figure:
    """Aplica um layout visual padronizado às figuras Plotly do notebook.

    Args:
        fig: Figura Plotly (Express ou Graph Objects) a ser formatada.
        title: Título exibido no topo da figura.
        height: Altura da figura em pixels.
        width: Largura da figura em pixels.

    Returns:
        go.Figure: A mesma figura recebida, com o layout já aplicado.

    Notes:
        Centraliza estilo (fonte, grade, legenda horizontal) para manter
        consistência visual entre todas as figuras do estudo.
    """
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=16)),
        template="plotly_white",
        font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=13),
        height=height,
        width=width,
        margin=dict(l=60, r=30, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        hovermode="x unified",
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    return fig

#pio.renderers.default = "notebook"
pio.renderers.default = "iframe"
def render_fig(fig: go.Figure) -> None:
    """Exibe uma figura Plotly.

    Args:
        fig: Figura Plotly a ser exibida.

    Returns:
        None.
    """
    try:
        fig.show(renderer=pio.renderers.default)
    except Exception:
        display(fig)

<a id="sec-problema-hipotese"></a>
## 1.2 Problema e Hipótese

Prever chuva é um problema naturalmente difícil. Mesmo em uma formulação binária, a relação entre variáveis meteorológicas e a ocorrência do evento pode variar ao longo do tempo por efeitos sazonais, mudanças de regime e ruído observacional.

A hipótese central deste experimento é direta: **em um problema temporal longo, estratégias com atualização do modelo devem se adaptar melhor do que uma abordagem estática**.

Com isso, a pergunta do estudo passa a ser:

> qual estratégia de atualização mantém melhor desempenho ao longo do tempo em uma regressão logística incremental aplicada a dados reais de precipitação?

<a id="sec-f1"></a>
### 1.2.1 Por que F1-score como Métrica Principal

Eventos de chuva são a classe minoritária neste conjunto de dados: a maior parte das horas não registra precipitação. Nesse cenário, a **acurácia é enganosa**. Um modelo que sempre prevê "sem chuva" já acertaria a maioria das observações sem capturar nenhum evento real.

Por isso a métrica de referência aqui é o **F1-score**, que equilibra:

- **Precisão** : dos eventos previstos como chuva, quantos de fato choveram;
- **Recall** : dos eventos reais de chuva, quantos o modelo conseguiu identificar.

O F1 penaliza modelos que sacrificam uma dimensão pela outra, o que o torna mais informativo do que a acurácia isolada para problemas desbalanceados como este.


In [5]:
# Constantes do experimento: nomes de colunas, limiar de decisão e
# configuração das janelas temporais deslizantes.
TARGET_RAW: str = "p01i"
TARGET_BINARY: str = "rain_event"
FEATURES: List[str] = ["tmpf", "dwpf", "relh", "drct", "sknt"]

THRESHOLD: float = 0.30
TRAIN_DAYS: int = 180
TEST_DAYS: int = 14
STEP_DAYS: int = 14
REGION: str = "MIA"

<a id="sec-metodologia"></a>
# 2. Metodologia

<a id="sec-cenarios"></a>
## 2.1 Cenários Comparados

Os três cenários abaixo usam a mesma sequência de janelas temporais. Isso garante comparabilidade metodológica e evita que diferenças de desempenho sejam explicadas por divisões distintas de treino e teste.

| Cenário | Como funciona | O que representa |
|---|---|---|
| Incremental | O modelo é treinado inicialmente e depois atualizado continuamente com novas observações. | Aprendizado contínuo com memória acumulada. |
| Retreinamento | O modelo é recriado do zero em cada nova janela de treino. | Atualização periódica sem memória entre janelas. |
| Treinamento único | O modelo é treinado uma única vez e reutilizado sem atualização. | Baseline estática para medir o valor prático da adaptação temporal. |

Antes de detalhar cada estratégia, vale destacar um ponto metodológico central: **os três cenários compartilham exatamente as mesmas janelas temporais, os mesmos dados e o mesmo processo de avaliação**. A única variável que muda entre eles é a política de atualização do modelo. Essa padronização é o que torna a comparação justa. Qualquer diferença de desempenho pode ser atribuída à estratégia de atualização, e não a diferenças na divisão dos dados.


In [6]:
@dataclass
class WindowConfig:
    """Configuração das janelas temporais deslizantes usadas na avaliação.

    Attributes:
        train_days: Quantidade de dias usados para treino em cada janela.
        test_days: Quantidade de dias usados para teste em cada janela.
        step_days: Deslocamento em dias entre o início de uma janela e a
            próxima.
    """
    train_days: int = TRAIN_DAYS
    test_days: int = TEST_DAYS
    step_days: int = STEP_DAYS


def make_model() -> compose.Pipeline:
    """Cria uma nova instância do pipeline de regressão logística incremental.

    Returns:
        compose.Pipeline: Pipeline do River com padronização de atributos
        seguida de regressão logística treinada via SGD.

    Notes:
        A mesma configuração de hiperparâmetros é usada em todos os
        cenários para garantir comparabilidade entre eles.
    """
    return compose.Pipeline(
        preprocessing.StandardScaler(),
        linear_model.LogisticRegression(
            optimizer=optim.SGD(lr=0.005),
            loss=optim.losses.Log(),
            l2=1e-4,
            intercept_lr=0.005,
            clip_gradient=1e12,
        ),
    )

<a id="sec-ambiente"></a>
## 2.2 Ambiente e Escolhas Metodológicas

O notebook foi estruturado para manter o fluxo reprodutível e interpretável. A preparação da base, a definição das janelas, a modelagem, a avaliação e a análise estatística foram mantidas em blocos separados.

Algumas decisões metodológicas merecem destaque:

- o problema foi tratado como classificação binária de chuva;
- a avaliação foi feita com **particionamento temporal**, e não aleatório;
- os dados foram mantidos em sua distribuição original, sem balanceamento artificial;
- a análise final prioriza **F1, precisão e recall**, métricas mais informativas em cenário desbalanceado.

<a id="sec-dataset"></a>
## 2.3 Dataset e Pré-processamento

<a id="sec-preparacao-dados"></a>
### 2.3.1 Preparação dos Dados

A base foi filtrada para a estação de interesse em Miami e convertida para um conjunto analítico limpo, ordenado no tempo e pronto para a modelagem.

Nesta etapa, foram tratados valores ausentes e códigos especiais, convertidos os atributos para formato numérico e criada a variável-alvo binária de ocorrência de chuva. A escolha foi manter uma linha de base simples e transparente, evitando intervenções que dificultassem a interpretação do modelo.

In [7]:
def prepare_region_data(asos_raw: pd.DataFrame) -> pd.DataFrame:
    """Filtra, limpa e enriquece os dados brutos da estação de interesse.

    Args:
        asos_raw: DataFrame bruto com observações meteorológicas de todas
            as estações da rede ASOS.

    Returns:
        pd.DataFrame: Conjunto analítico filtrado para a estação
        definida em REGION, ordenado temporalmente, com a variável-alvo
        binária de chuva e colunas auxiliares de data já calculadas.

    Raises:
        ValueError: Caso o DataFrame resultante fique vazio após a
            limpeza dos dados.

    Notes:
        Códigos especiais de ausência ("M") e traço de chuva ("T") são
        tratados antes da conversão para tipos numéricos, preservando a
        semântica original dos dados meteorológicos.
    """
    df = asos_raw.copy()
    df = df[df["station"] == REGION].copy()

    df[TARGET_RAW] = (
        df[TARGET_RAW]
        .astype(str)
        .str.strip()
        .replace({"M": np.nan, "": np.nan, "T": "0.00"})
    )
    df[TARGET_RAW] = pd.to_numeric(df[TARGET_RAW], errors="coerce")

    for col in FEATURES:
        df[col] = df[col].astype(str).str.strip().replace({"M": np.nan, "": np.nan})
        if col == "drct":
            df[col] = df[col].replace({"999": np.nan, "VRB": np.nan})
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["valid"] = pd.to_datetime(df["valid"], utc=True, errors="coerce")
    df = df.dropna(subset=FEATURES + [TARGET_RAW, "valid"]).copy()
    df = df.sort_values("valid").reset_index(drop=True)

    if df.empty:
        raise ValueError(
            f"Nenhum registro válido encontrado para a estação '{REGION}' "
            "após a limpeza dos dados."
        )

    df[TARGET_BINARY] = df[TARGET_RAW] >= 0.01
    df["year"] = df["valid"].dt.year
    df["month"] = df["valid"].dt.month
    df["year_month"] = df["valid"].dt.tz_convert(None).dt.to_period("M").astype(str)

    min_valid = df["valid"].min()
    df["month_seq"] = 12 * (df["year"] - min_valid.year) + (df["month"] - min_valid.month) + 1
    df["day_seq"] = (df["valid"] - min_valid).dt.days + 1

    keep_cols = [
        "station", "valid", TARGET_RAW, TARGET_BINARY, *FEATURES,
        "year", "month", "year_month", "month_seq", "day_seq"
    ]
    return df[keep_cols].copy()

<a id="sec-validacao-temporal"></a>
### 2.3.2 Estratégia de Validação Temporal

Como o problema tem natureza temporal, a avaliação não usa particionamento aleatório. Em vez disso, o experimento adota **janelas deslizantes**, sempre treinando no passado e testando em um período futuro imediato.

Cada iteração usa:

- 180 dias para treino;
- 14 dias para teste;
- avanço de 14 dias entre janelas.

Essa configuração preserva sequência temporal e aproxima melhor o experimento de um cenário real de uso.

In [8]:
def create_flexible_windows(
    df: pd.DataFrame,
    config: WindowConfig,
) -> Tuple[pd.DataFrame, List[Dict]]:
    """Gera janelas deslizantes de treino/teste sobre a série temporal.

    Args:
        df: DataFrame ordenado temporalmente, contendo a coluna
            auxiliar "day_seq" produzida por `prepare_region_data`.
        config: Configuração com os tamanhos de treino, teste e passo
            entre janelas consecutivas.

    Returns:
        Tuple[pd.DataFrame, List[Dict]]: O DataFrame reordenado e a
        lista de janelas geradas, cada uma com os índices de treino e
        teste e as respectivas datas de início e fim.

    Notes:
        A janela avança sempre no tempo (treino no passado, teste no
        futuro imediato), preservando a sequência temporal exigida
        por um experimento de série temporal.
    """
    df = df.sort_values("valid").reset_index(drop=True).copy()
    windows = []

    current_seq = int(df["day_seq"].min())
    max_seq = int(df["day_seq"].max())

    while current_seq + config.train_days + config.test_days - 1 <= max_seq:
        train_start_seq = current_seq
        train_end_seq = current_seq + config.train_days - 1
        test_start_seq = train_end_seq + 1
        test_end_seq = test_start_seq + config.test_days - 1

        train_mask = (df["day_seq"] >= train_start_seq) & (df["day_seq"] <= train_end_seq)
        test_mask = (df["day_seq"] >= test_start_seq) & (df["day_seq"] <= test_end_seq)

        windows.append({
            "train_idx": df.index[train_mask].tolist(),
            "test_idx": df.index[test_mask].tolist(),
            "train_start_date": df.loc[train_mask, "valid"].iloc[0],
            "train_end_date": df.loc[train_mask, "valid"].iloc[-1],
            "test_start_date": df.loc[test_mask, "valid"].iloc[0],
            "test_end_date": df.loc[test_mask, "valid"].iloc[-1],
        })

        current_seq += config.step_days

    return df, windows


def validate_windows(df: pd.DataFrame, windows: List[Dict]) -> bool:
    """Valida a integridade temporal das janelas geradas.

    Args:
        df: DataFrame de referência usado na geração das janelas.
        windows: Lista de janelas produzidas por
            `create_flexible_windows`.

    Returns:
        bool: True se todas as janelas forem válidas.

    Raises:
        AssertionError: Caso alguma janela possua sobreposição entre os
            índices de treino e teste, ou caso o período de treino não
            anteceda integralmente o período de teste.

    Notes:
        Essa checagem existe para evitar vazamento de dados (data
        leakage) entre treino e teste, garantindo sequência temporal.
    """
    for w in windows:
        tr = df.loc[w["train_idx"]]
        te = df.loc[w["test_idx"]]
        assert set(w["train_idx"]).isdisjoint(set(w["test_idx"]))
        assert tr["valid"].max() < te["valid"].min()
    return f"{len(windows)} janelas validadas com sucesso. Treino e teste estão temporalmente consistentes."

<a id="sec-metricas"></a>
## 2.4 Métricas de Avaliação

Em vez de armazenar apenas métricas agregadas, o notebook registra os componentes da matriz de confusão em cada janela. Isso permite recalcular precisão, recall e F1 em diferentes níveis de agregação, como por janela, por ano ou no total do experimento.

Essa escolha é importante porque o problema apresenta desbalanceamento. Nesse contexto, **acurácia isolada pode ser enganosa**, enquanto F1, precisão e recall descrevem melhor o comportamento do modelo diante da classe positiva.

In [9]:
def predict_with_threshold(
    model,
    X_df: pd.DataFrame,
    threshold: float = 0.5,
) -> Tuple[np.ndarray, np.ndarray]:
    """Gera previsões binárias a partir das probabilidades do modelo.

    Args:
        model: Modelo incremental compatível com a API do River
            (deve expor `predict_proba_one`).
        X_df: DataFrame com as features de entrada, uma linha por
            observação.
        threshold: Limiar de decisão aplicado sobre a probabilidade da
            classe positiva.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Array booleano de previsões e
        array de probabilidades associadas à classe positiva.

    Notes:
        A previsão é feita amostra por amostra, respeitando a natureza
        online do modelo incremental.
    """
    probs, preds = [], []

    for x in X_df.to_dict("records"):
        proba_dict = model.predict_proba_one(x)
        p1 = proba_dict.get(True, 0.0)
        probs.append(p1)
        preds.append(p1 >= threshold)

    return np.array(preds, dtype=bool), np.array(probs)


def confusion_summary(y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    """Calcula os componentes da matriz de confusão binária.

    Args:
        y_true: Array booleano com os rótulos reais.
        y_pred: Array booleano com os rótulos previstos.

    Returns:
        Dict: Dicionário com as contagens de verdadeiros negativos
        (tn), falsos positivos (fp), falsos negativos (fn) e
        verdadeiros positivos (tp).
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def compute_metrics_from_confusion(tp, tn, fp, fn) -> pd.DataFrame:
    """Deriva acurácia, precisão, recall e F1 a partir da matriz de confusão.

    Args:
        tp: Contagem(s) de verdadeiros positivos.
        tn: Contagem(s) de verdadeiros negativos.
        fp: Contagem(s) de falsos positivos.
        fn: Contagem(s) de falsos negativos.

    Returns:
        pd.DataFrame: DataFrame com as colunas "accuracy", "precision",
        "recall" e "f1", calculadas de forma vetorizada.

    Notes:
        Divisões por zero são tratadas com `np.where`, retornando 0.0
        para janelas sem suporte suficiente, em vez de gerar erro.
    """
    total = tp + tn + fp + fn
    accuracy = np.where(total > 0, (tp + tn) / total, 0.0)
    precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1 = np.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)

    return pd.DataFrame({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })


def evaluate_model_on_window(
    model,
    test_data: pd.DataFrame,
    threshold: float = 0.5,
) -> Dict:
    """Avalia um modelo sobre a janela de teste e mede o tempo de inferência.

    Args:
        model: Modelo incremental já treinado, compatível com o River.
        test_data: DataFrame da janela de teste, contendo as features e
            a variável-alvo binária.
        threshold: Limiar de decisão usado na binarização das
            previsões.

    Returns:
        Dict: Métricas de confusão, suporte da classe positiva, número
        de amostras, tempo de inferência e os arrays de rótulos reais,
        previstos e probabilidades.
    """
    
    y_true = test_data[TARGET_BINARY].astype(bool).to_numpy()
    y_pred, y_prob = predict_with_threshold(model, test_data[FEATURES], threshold)

    metrics = confusion_summary(y_true, y_pred)

    class1_count = int(np.sum(y_true))
    metrics.update({
        "class1_support": class1_count,
        "class1_in_test": class1_count > 0,
        "n_test_samples": len(y_true),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    })
    return metrics

<a id="sec-estrutura-resultados"></a>
## 2.5 Estrutura dos Resultados

Cada janela gera uma linha padronizada com intervalo temporal, componentes da matriz de confusão, suporte da classe positiva e artefatos de predição. Essa padronização facilita a auditoria do experimento e torna a análise posterior mais flexível.

In [10]:
def build_result_row(
    window_idx: int,
    window_info: Dict,
    metrics_info: Dict,
    scenario: str,
) -> Dict:
    """Monta uma linha padronizada de resultado para uma janela avaliada.

    Args:
        window_idx: Índice sequencial da janela dentro do cenário.
        window_info: Dicionário com as datas de treino e teste da
            janela, gerado por `create_flexible_windows`.
        metrics_info: Dicionário de métricas retornado por
            `evaluate_model_on_window`.
        scenario: Nome do cenário avaliado (ex.: "Incremental").

    Returns:
        Dict: Registro consolidado, pronto para compor o DataFrame de
        resultados do experimento.
    """
    return {
        "window_idx": window_idx,
        "scenario": scenario,
        "train_start": str(window_info["train_start_date"]),
        "train_end": str(window_info["train_end_date"]),
        "test_start": str(window_info["test_start_date"]),
        "test_end": str(window_info["test_end_date"]),
        "tp": metrics_info["tp"],
        "tn": metrics_info["tn"],
        "fp": metrics_info["fp"],
        "fn": metrics_info["fn"],
        "class1_support": metrics_info["class1_support"],
        "class1_in_test": metrics_info["class1_in_test"],
        "n_test_samples": metrics_info["n_test_samples"],
        "y_true": metrics_info["y_true"],
        "y_pred": metrics_info["y_pred"],
        "y_prob": metrics_info["y_prob"],
    }

<a id="sec-incremental"></a>
## 2.6 Cenário 1 — Aprendizado Incremental

No primeiro cenário, o modelo é treinado na janela inicial e depois atualizado continuamente com as observações mais recentes antes de cada nova previsão.

Esse cenário representa um fluxo de aprendizado contínuo, no qual o modelo preserva memória acumulada ao longo do tempo e se ajusta gradualmente à evolução da série.

In [11]:
def run_scenario_incremental(
    df: pd.DataFrame,
    windows: List[Dict],
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Executa o cenário de aprendizado incremental sobre todas as janelas.

    O modelo é treinado uma vez na primeira janela e, em seguida,
    atualizado continuamente com os dados de teste de cada janela
    anterior antes de prever a janela seguinte.

    Args:
        df: DataFrame preparado, contendo as colunas de features e a
            variável-alvo binária.
        windows: Lista de janelas temporais geradas por
            `create_flexible_windows`.
        threshold: Limiar de decisão usado na binarização das
            previsões.

    Returns:
        pd.DataFrame: Um registro por janela avaliada, no formato
        produzido por `build_result_row`.

    Notes:
        Esta função utiliza aprendizado online (`learn_one`) e mantém
        memória acumulada do modelo entre janelas consecutivas.
    """
    rows = []
    model = make_model()
    scenario = "Incremental"

    w0 = windows[0]
    train0 = df.loc[w0["train_idx"]].copy()
    test0 = df.loc[w0["test_idx"]].copy()

    for x, y in zip(train0[FEATURES].to_dict("records"), train0[TARGET_BINARY].astype(bool)):
        model.learn_one(x, y)

    metrics_info = evaluate_model_on_window(model, test0, threshold)
    rows.append(build_result_row(1, w0, metrics_info, scenario))

    previous_test = test0.copy()

    for i, w in enumerate(windows[1:], start=2):
        for x, y in zip(previous_test[FEATURES].to_dict("records"), previous_test[TARGET_BINARY].astype(bool)):
            model.learn_one(x, y)

        current_test = df.loc[w["test_idx"]].copy()
        metrics_info = evaluate_model_on_window(model, current_test, threshold)
        rows.append(build_result_row(i, w, metrics_info, scenario))
        previous_test = current_test.copy()

    return pd.DataFrame(rows)

<a id="sec-retreinamento"></a>
## 2.7 Cenário 2 — Retreinamento Periódico

No segundo cenário, o modelo é reconstruído do zero em cada janela temporal, usando apenas os dados disponíveis no bloco de treino correspondente.

Essa estratégia representa uma forma mais conservadora de adaptação: ela incorpora dados recentes, mas não mantém memória direta do histórico anterior entre janelas.

In [12]:
def run_scenario_retrain(
    df: pd.DataFrame,
    windows: List[Dict],
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Executa o cenário de retreinamento periódico sobre todas as janelas.

    Em cada janela, um novo modelo é criado do zero e treinado apenas
    com os dados de treino daquela janela específica, sem herdar
    conhecimento das janelas anteriores.

    Args:
        df: DataFrame preparado, contendo as colunas de features e a
            variável-alvo binária.
        windows: Lista de janelas temporais geradas por
            `create_flexible_windows`.
        threshold: Limiar de decisão usado na binarização das
            previsões.

    Returns:
        pd.DataFrame: Um registro por janela avaliada, no formato
        produzido por `build_result_row`.

    Notes:
        Representa uma adaptação sem memória entre janelas: cada
        modelo "esquece" o histórico anterior ao ser recriado.
    """
    rows = []
    scenario = "Retreinamento"

    for i, w in enumerate(windows, start=1):
        train_data = df.loc[w["train_idx"]].copy()
        test_data = df.loc[w["test_idx"]].copy()

        model = make_model()
        for x, y in zip(train_data[FEATURES].to_dict("records"), train_data[TARGET_BINARY].astype(bool)):
            model.learn_one(x, y)

        metrics_info = evaluate_model_on_window(model, test_data, threshold)
        rows.append(build_result_row(i, w, metrics_info, scenario))

    return pd.DataFrame(rows)

<a id="sec-treinamento-unico"></a>
## 2.8 Cenário 3 — Treinamento Único

No terceiro cenário, o modelo é treinado apenas uma vez na primeira janela e depois reutilizado para prever todas as janelas futuras sem qualquer atualização adicional.

Esse cenário funciona como baseline estática e ajuda a medir o valor prático da adaptação temporal.

In [13]:
def run_scenario_static(
    df: pd.DataFrame,
    windows: List[Dict],
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Executa o cenário de treinamento único (baseline estática).

    O modelo é treinado apenas uma vez, na primeira janela, e depois
    reutilizado sem qualquer atualização para prever todas as janelas
    subsequentes.

    Args:
        df: DataFrame preparado, contendo as colunas de features e a
            variável-alvo binária.
        windows: Lista de janelas temporais geradas por
            `create_flexible_windows`.
        threshold: Limiar de decisão usado na binarização das
            previsões.

    Returns:
        pd.DataFrame: Um registro por janela avaliada, no formato
        produzido por `build_result_row`.

    Notes:
        Serve como referência estática para medir o valor prático da
        adaptação temporal dos demais cenários.
    """
    rows = []
    model = make_model()
    scenario = "Treinamento único"

    w0 = windows[0]
    train0 = df.loc[w0["train_idx"]].copy()

    for x, y in zip(train0[FEATURES].to_dict("records"), train0[TARGET_BINARY].astype(bool)):
        model.learn_one(x, y)

    for i, w in enumerate(windows, start=1):
        test_data = df.loc[w["test_idx"]].copy()
        metrics_info = evaluate_model_on_window(model, test_data, threshold)
        rows.append(build_result_row(i, w, metrics_info, scenario))

    return pd.DataFrame(rows)

<a id="sec-execucao"></a>
# 3. Execução do Experimento

Com as funções definidas, o experimento começa pelo carregamento da base bruta, preparação dos dados e geração das janelas temporais. Em seguida, os três cenários são executados com exatamente a mesma estrutura de avaliação.

In [14]:
# Carrega a base bruta, prepara os dados e gera as janelas temporais.
DATA_PATH = "MIA_2012_2025.csv"

asos_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = prepare_region_data(asos_raw)
df, windows = create_flexible_windows(df, WindowConfig())
print(validate_windows(df, windows))

# Executa os três cenários usando exatamente a mesma estrutura de janelas.
results_incremental = run_scenario_incremental(df, windows, THRESHOLD)
results_retrain = run_scenario_retrain(df, windows, THRESHOLD)
results_static = run_scenario_static(df, windows, THRESHOLD)

results = pd.concat([results_incremental, results_retrain, results_static], ignore_index=True)
results["test_start"] = pd.to_datetime(results["test_start"], utc=True, errors="coerce")
results["year"] = results["test_start"].dt.year

350 janelas validadas com sucesso. Treino e teste estão temporalmente consistentes.


<a id="sec-resultados"></a>
# 4. Resultados

<a id="sec-dinamica-alvo"></a>
## 4.1 Dinâmica Temporal da Variável-Alvo

Antes de comparar os cenários, vale observar o comportamento da própria série ao longo do período. Essa leitura ajuda a identificar sazonalidade, amplitude de variação e mudanças de regime que podem afetar diretamente o desempenho dos modelos.

Além disso, o conjunto apresenta desbalanceamento entre classes, mas a análise foi mantida sobre a distribuição original dos dados. Isso torna os resultados mais próximos de um cenário real de aplicação.

In [15]:
monthly_precip = (
    df.groupby("year_month", as_index=False)
      .agg(hours=("rain_event", "size"), rainy_hours=("rain_event", "sum"))
)
monthly_precip["year_month_dt"] = pd.to_datetime(monthly_precip["year_month"])
monthly_precip["pct_rain_hours"] = 100 * monthly_precip["rainy_hours"] / monthly_precip["hours"]
monthly_precip["year"] = monthly_precip["year_month_dt"].dt.year
monthly_precip["month"] = monthly_precip["year_month_dt"].dt.month

In [16]:
fig = px.line(
    monthly_precip.sort_values("year_month_dt"),
    x="year_month_dt",
    y="pct_rain_hours",
    line_shape="spline",
    markers=False,
)
fig.update_traces(line=dict(color="#1f6feb", width=1.6), hovertemplate="Mês: %{x|%Y-%m}<br>%% chuva: %{y:.1f}%<extra></extra>")
fig.update_xaxes(title_text="Ano")
fig.update_yaxes(title_text="% de horas com chuva")
apply_layout(fig, "Figura 1. Percentual mensal de horas com chuva ao longo do período")
render_fig(fig)


**Interpretação:** o percentual mensal de horas com chuva mostra um padrão sazonal claro, com picos recorrentes que se repetem ano a ano, compatível com o regime de chuvas de verão típico da região. Essa sazonalidade é justamente o tipo de variação que um modelo estático não consegue acompanhar, e é o pano de fundo que motiva a comparação entre as três estratégias de atualização.


<a id="sec-consolidacao"></a>
## 4.2 Consolidação dos Resultados

Após a execução dos cenários, os resultados são consolidados em dois níveis principais: resumo global por cenário e resumo anual por cenário. Essa separação permite comparar não apenas o desempenho médio, mas também a estabilidade temporal de cada estratégia.

In [17]:
summary = (
    results.groupby("scenario", as_index=False)
           .agg(tp=("tp", "sum"), tn=("tn", "sum"), fp=("fp", "sum"), fn=("fn", "sum"), windows=("window_idx", "count"))
)
summary = pd.concat([summary, compute_metrics_from_confusion(summary["tp"], summary["tn"], summary["fp"], summary["fn"])], axis=1)

yearly_results = (
    results.groupby(["scenario", "year"], as_index=False)
           .agg(tp=("tp", "sum"), tn=("tn", "sum"), fp=("fp", "sum"), fn=("fn", "sum"), windows=("window_idx", "count"))
)
yearly_metrics = compute_metrics_from_confusion(yearly_results["tp"], yearly_results["tn"], yearly_results["fp"], yearly_results["fn"])
yearly_results = pd.concat([yearly_results, yearly_metrics], axis=1)

<a id="sec-resultados-agregados"></a>
## 4.3 Leitura dos Resultados Agregados

O resumo global oferece uma visão inicial de desempenho, mas ele não deve ser interpretado isoladamente. Em séries temporais longas, o comportamento anual importa tanto quanto a média final, porque um cenário pode parecer competitivo no agregado e ainda assim falhar em estabilidade ao longo do tempo.

Por isso, a leitura principal deste estudo se apoia na evolução anual do F1 e nas comparações estatísticas pareadas entre os cenários.

In [18]:
display(summary.sort_values("f1", ascending=False))

,scenario,tp,tn,fp,fn,windows,accuracy,precision,recall,f1
0,Incremental,5204,111390,5594,6143,350,0.908541,0.481941,0.458623,0.469993
1,Retreinamento,4080,111011,5973,7267,350,0.896829,0.405849,0.359566,0.381308
2,Treinamento único,6624,95128,21856,4723,350,0.792887,0.232584,0.583767,0.332639


**Tabela 1.** Resumo global por cenário (acurácia, precisão, recall e F1 agregados em todas as janelas).

**Interpretação:** considerando o F1 calculado sobre o conjunto agregado de todas as janelas avaliadas, o cenário **Incremental** apresenta o maior valor entre os três. O cenário **Treinamento único** chama atenção pela combinação de recall elevado (0,584) com precisão baixa (0,233), indicando tendência a prever chuva em excesso à medida que a distribuição dos dados se afasta da observada no treinamento original. Esse resultado agregado, porém, não captura a evolução temporal do desempenho; por isso, a próxima seção analisa o comportamento anual de cada estratégia.


In [19]:
fig = go.Figure()
for scenario, group in yearly_results.sort_values("year").groupby("scenario"):
    fig.add_trace(go.Scatter(
        x=group["year"],
        y=group["f1"],
        mode="lines+markers",
        name=scenario,
        line=dict(shape="spline", smoothing=0.4, width=2.5, color=SCENARIO_COLORS.get(scenario)),
        marker=dict(size=7, color=SCENARIO_COLORS.get(scenario)),
        hovertemplate="Ano: %{x}<br>F1: %{y:.3f}<extra>" + scenario + "</extra>",
    ))

fig.update_xaxes(title_text="Ano", dtick=1)
fig.update_yaxes(title_text="F1-score")
apply_layout(fig, "Figura 2. Evolução anual do F1-score por cenário")
render_fig(fig)


**Interpretação:** a evolução anual confirma o padrão observado no resumo agregado. O cenário **Incremental** apresenta o maior F1 em todos os anos avaliados, superando de forma consistente tanto o **Retreinamento** quanto o **Treinamento Único**. O **Retreinamento** ocupa sistematicamente uma posição intermediária, reduzindo parte da degradação observada no **Treinamento Único**, mas permanecendo abaixo do desempenho alcançado pelo aprendizado **Incremental**. Essa consistência ao longo de toda a série reforça que a vantagem do cenário incremental não decorre de poucos anos específicos, mas representa um comportamento recorrente ao longo do período analisado.


In [20]:
order = ["Treinamento único", "Retreinamento", "Incremental"]

fig = go.Figure()
for scenario in order:
    values = yearly_results.loc[yearly_results["scenario"] == scenario, "f1"]
    fig.add_trace(go.Box(
        y=values,
        name=scenario,
        marker_color=SCENARIO_COLORS.get(scenario),
        boxmean=False,
        boxpoints="all",
        jitter=0.4,
        pointpos=0,
        hovertemplate="F1: %{y:.3f}<extra>" + scenario + "</extra>",
    ))

fig.update_yaxes(title_text="F1-score")
apply_layout(fig, "Figura 3. Dispersão do F1 anual por cenário", width=800)
fig.update_layout(showlegend=False)
render_fig(fig)


**Interpretação:** o boxplot resume simultaneamente o nível de desempenho e sua variabilidade ao longo dos anos. O cenário **Incremental** apresenta a maior mediana de F1, indicando desempenho típico superior. O **Treinamento único** concentra valores em um patamar mais baixo, refletindo a degradação observada ao longo da série temporal. Já o **Retreinamento** ocupa uma posição intermediária, com variabilidade comparável à do cenário incremental. Essa análise de distribuição complementa a comparação temporal e motiva o teste estatístico pareado apresentado a seguir.


In [21]:
diff_pivot = yearly_results.pivot(index="year", columns="scenario", values="f1").sort_index()
diff_series = (diff_pivot["Incremental"] - diff_pivot["Retreinamento"]).dropna()
bar_colors = ["#2ca02c" if v >= 0 else "#d62728" for v in diff_series.values]

fig = go.Figure(go.Bar(
    x=diff_series.index.astype(str),
    y=diff_series.values,
    marker_color=bar_colors,
    text=[f"{v:.2f}" for v in diff_series.values],
    textposition="outside",
    hovertemplate="Ano: %{x}<br>Δ F1: %{y:.3f}<extra></extra>",
))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_xaxes(title_text="Ano")
fig.update_yaxes(title_text="Δ F1")
apply_layout(fig, "Figura 4. Diferença anual de F1 (Incremental − Retreinamento)")
render_fig(fig)


**Interpretação:** a diferença anual de F1 entre **Incremental** e **Retreinamento** é positiva em todos os anos avaliados, indicando que a vantagem do aprendizado incremental não depende de poucos anos atípicos, mas se manifesta de forma recorrente ao longo de toda a série temporal.


<a id="sec-comparacao-estatistica"></a>
## 4.4 Comparação Estatística entre Cenários

Para complementar a análise visual, foi aplicado o teste de Wilcoxon pareado sobre os valores anuais de F1. A escolha desse teste é adequada porque a comparação é feita entre observações anuais pareadas, sem assumir normalidade das diferenças.

As comparações foram realizadas entre os três pares de cenários, com correção de Bonferroni para controlar erro por múltiplos testes. Além do valor-p ajustado, também foi calculado o tamanho de efeito rr, que ajuda a interpretar a relevância prática das diferenças.

In [22]:
def interpret_effect_size(r: float) -> str:
    """Classifica a magnitude do tamanho de efeito r em categorias qualitativas.

    Args:
        r: Tamanho de efeito calculado a partir da estatística z do
            teste de Wilcoxon.

    Returns:
        str: Uma das categorias "muito pequeno", "pequeno", "moderado",
        "grande" ou "indefinido" (quando r é NaN).
    """
    if pd.isna(r):
        return "indefinido"
    if abs(r) < 0.10:
        return "muito pequeno"
    if abs(r) < 0.30:
        return "pequeno"
    if abs(r) < 0.50:
        return "moderado"
    return "grande"


def wilcoxon_with_effect_size(
    yearly_results: pd.DataFrame,
    group_a: str,
    group_b: str,
    alpha: float = 0.05,
    n_comparisons: int = 1,
) -> Dict:
    """Compara dois cenários via teste de Wilcoxon pareado sobre o F1 anual.

    Args:
        yearly_results: DataFrame com o F1 anual por cenário, contendo
            ao menos as colunas "year", "scenario" e "f1".
        group_a: Nome do primeiro cenário a comparar.
        group_b: Nome do segundo cenário a comparar.
        alpha: Nível de significância antes da correção de Bonferroni.
        n_comparisons: Número total de comparações realizadas, usado
            para ajustar o alpha via correção de Bonferroni.

    Returns:
        Dict: Estatística do teste, valor-p bruto e ajustado, alpha
        ajustado, decisão sobre rejeitar H0, tamanho de efeito r e sua
        interpretação qualitativa.

    Notes:
        O tamanho de efeito r é derivado da estatística z aproximada do
        teste de Wilcoxon, dividida pela raiz do número de pares não
        nulos, seguindo a convenção usual para esse teste.
    """
    pivot = (
        yearly_results
        .pivot(index="year", columns="scenario", values="f1")
        .dropna(subset=[group_a, group_b])
    )

    x = pivot[group_a].to_numpy()
    y = pivot[group_b].to_numpy()
    test = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided", method="approx")

    diff = x - y
    diff_nz = diff[diff != 0]
    n = len(diff_nz)

    if n == 0:
        r = np.nan
    else:
        ranks = pd.Series(np.abs(diff_nz)).rank(method="average").to_numpy()
        w_plus = np.sum(ranks[diff_nz > 0])
        mu_w = n * (n + 1) / 4
        sigma_w = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        z = (w_plus - mu_w) / sigma_w if sigma_w > 0 else np.nan
        r = z / np.sqrt(n) if sigma_w > 0 else np.nan

    alpha_bonf = alpha / n_comparisons

    return {
        "comparison": f"{group_a} vs {group_b}",
        "n_pairs": len(pivot),
        "wilcoxon_stat": float(test.statistic),
        "pvalue": float(test.pvalue),
        "pvalue_bonferroni": min(float(test.pvalue) * n_comparisons, 1.0),
        "alpha_bonferroni": alpha_bonf,
        "reject_h0": float(test.pvalue) < alpha_bonf,
        "r": float(r) if pd.notna(r) else np.nan,
        "effect_magnitude": interpret_effect_size(r),
    }

<a id="sec-interpretacao-estatistica"></a>
## 4.5 Interpretação Estatística

A interpretação estatística deve considerar dois níveis. O primeiro é a significância, indicada pelo valor-p ajustado. O segundo é a relevância prática, indicada pelo tamanho de efeito.

Essa combinação evita conclusões superficiais do tipo “foi significativo, logo é melhor”. Em aplicações reais, diferenças pequenas podem ser estatisticamente detectáveis sem representar ganho operacional relevante.

In [23]:
paired_f1 = yearly_results.pivot(index="year", columns="scenario", values="f1").sort_index()
comparisons = [
    ("Incremental", "Treinamento único"),
    ("Incremental", "Retreinamento"),
    ("Retreinamento", "Treinamento único"),
]
stat_results = pd.DataFrame([
    wilcoxon_with_effect_size(yearly_results, a, b, alpha=0.05, n_comparisons=len(comparisons))
    for a, b in comparisons
])

display(stat_results)

,comparison,n_pairs,wilcoxon_stat,pvalue,pvalue_bonferroni,alpha_bonferroni,reject_h0,r,effect_magnitude
0,Incremental vs Treinamento único,14,0.0,0.000982,0.002945,0.016667,True,0.880830,grande
1,Incremental vs Retreinamento,14,0.0,0.000982,0.002945,0.016667,True,0.880830,grande
2,Retreinamento vs Treinamento único,14,8.0,0.005213,0.015640,0.016667,True,0.746609,grande


**Tabela 2.** Comparações pareadas de Wilcoxon sobre o F1 anual, com correção de Bonferroni e tamanho de efeito rr.

**Interpretação:** no teste de Wilcoxon pareado, rejeitar H0 indica que há evidência estatística de que a mediana das diferenças anuais de F1 entre os cenários é diferente de zero. Todas as comparações apresentaram diferenças estatisticamente significativas e efeito grande, confirmando a hierarquia de desempenho observada nas análises anteriores (**Incremental** > **Retreinamento** > **Treinamento Único**).


<a id="sec-discussao"></a>
# 5. Discussão

Os resultados indicam que a estratégia incremental tende a manter desempenho mais consistente ao longo do tempo. Isso sugere que a dinâmica temporal da série é suficientemente relevante para penalizar abordagens estáticas.

O cenário de treinamento único funciona como referência importante: ele mostra o que acontece quando o modelo não acompanha a evolução da distribuição. Já o retreinamento periódico oferece adaptação parcial, mas sem a mesma continuidade de ajuste observada no cenário incremental.

Do ponto de vista técnico, isso reforça uma ideia central deste trabalho: **em dados temporais reais, a estratégia de atualização do modelo faz diferença prática no resultado final**.

<a id="sec-limitacoes"></a>
## 5.1 Limitações do Experimento

Este estudo foi construído para ser claro e interpretável, mas algumas limitações devem ser explicitadas:

- o experimento usa uma única família de modelo;
- a comparação depende da configuração adotada para as janelas temporais;
- a análise anual reduz a granularidade temporal da comparação estatística;
- o conjunto foi mantido desbalanceado, o que reflete melhor o cenário real, mas aumenta a dificuldade do problema.

Essas limitações não invalidam o experimento. Pelo contrário: elas ajudam a contextualizar os resultados e deixam mais claro o alcance das conclusões.

<a id="sec-principais-resultados"></a>
## 5.2 Principais Resultados

- O cenário **Incremental** apresentou o maior F1 agregado (0,470) entre os três cenários.
- O cenário **Treinamento único** apresentou o menor F1 agregado (0,333) e evidências de degradação ao longo do tempo.
- O **Retreinamento periódico** melhorou o desempenho em relação ao treinamento único (F1 = 0,381), mas permaneceu abaixo do cenário incremental.
- Os resultados do teste de Wilcoxon reforçam a superioridade do cenário **Incremental** e o desempenho intermediário do **Retreinamento** em relação ao **Treinamento Único**.


<a id="sec-conclusao"></a>
# 6. Conclusão

Este notebook comparou três estratégias de atualização de modelo em uma tarefa de previsão binária de chuva horária em série temporal real. A análise combinou leitura temporal da base, avaliação por janelas, consolidação anual e teste estatístico pareado.

De forma geral, os resultados mostram que o aprendizado **Incremental** apresentou o melhor desempenho médio e a maior regularidade temporal de F1 entre os cenários avaliados. O **Treinamento Único** mostrou menor capacidade de adaptação ao longo do tempo, enquanto o **Retreinamento** periódico ocupou uma posição intermediária.

Mais do que apontar um vencedor, o principal resultado deste trabalho é mostrar que, em problemas temporais reais, **a política de atualização do modelo é parte central da solução**.